# ST-OMR Meter V5-3A HOLD report recovery

This cell does not refit a candidate. It reads and preserves the existing HOLD report, exposes both specialist reasons, and writes a separate recovery envelope.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import subprocess
import sys

EXPECTED_EXECUTION_HEAD = "cdc6683a556c16b00e7b154fca8e89ba5dd848b7"
SOURCE_HARNESS_HEAD = "c2d5f1652adac52387e33b9d2f33078f864f980b"
SOURCE_IMPLEMENTATION_CI_RUN_ID = 32735656612
SOURCE_HARNESS_CI_RUN_ID = 32737194764
REPOSITORY = "khfy7wpr5p-maker/st-omr-training"
REPO_URL = f"https://github.com/{REPOSITORY}.git"
REPO = Path("/content/st-omr-training")
MYDRIVE = Path("/content/drive/MyDrive")

if not MYDRIVE.is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
DATA_ROOT = MYDRIVE / "TEST" / "METER_V2_1500_PACKAGE_AB_CLEAN"
ANN_DIR = DATA_ROOT / "annotations"
if not ANN_DIR.is_dir():
    raise RuntimeError(f"annotations directory missing: {ANN_DIR}")
print("DRIVE CHECK = PASS")

if not REPO.exists():
    subprocess.check_call(["git", "clone", "--no-checkout", REPO_URL, str(REPO)])
elif not (REPO / ".git").is_dir():
    raise RuntimeError(f"REPO git repository degil: {REPO}")
remotes = subprocess.check_output(["git", "-C", str(REPO), "remote"], text=True).split()
if "origin" not in remotes:
    subprocess.check_call(["git", "-C", str(REPO), "remote", "add", "origin", REPO_URL])
else:
    subprocess.check_call(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL])
subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", EXPECTED_EXECUTION_HEAD, "--depth", "1"])
fetched_head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "FETCH_HEAD"], text=True).strip()
if fetched_head != EXPECTED_EXECUTION_HEAD:
    raise RuntimeError(f"FETCH_HEAD mismatch: {fetched_head}")
subprocess.check_call(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_EXECUTION_HEAD])
actual_head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
if actual_head != EXPECTED_EXECUTION_HEAD:
    raise RuntimeError(f"HEAD mismatch: {actual_head}")
if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip():
    raise RuntimeError("Repository worktree temiz degil")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from st_omr_training import meter_v5_1_bbox_pilot as v51
from st_omr_training import meter_v5_3a_robust_margin_head_candidate_v1 as repair
print("EXACT SOURCE CHECK = PASS")
print("EXECUTION HEAD =", actual_head)
print("SOURCE HARNESS HEAD =", SOURCE_HARNESS_HEAD)

REPORT_PATH = ANN_DIR / repair.REPORT_NAME
CANDIDATE_DIR = ANN_DIR / repair.CANDIDATE_DIR_NAME
TEMP_CANDIDATE_DIR = ANN_DIR / repair.TEMP_CANDIDATE_DIR_NAME
RECOVERY_ENVELOPE_PATH = ANN_DIR / f"v5_3a_render_recovery_envelope_{SOURCE_HARNESS_HEAD}.json"
if not REPORT_PATH.is_file():
    raise RuntimeError(f"Existing V5-3A report missing: {REPORT_PATH}")
if RECOVERY_ENVELOPE_PATH.exists():
    raise RuntimeError(f"Refusing recovery-envelope overwrite/rerun: {RECOVERY_ENVELOPE_PATH}")
if CANDIDATE_DIR.exists() or TEMP_CANDIDATE_DIR.exists():
    raise RuntimeError("candidate directory exists after HOLD")

source_bytes = REPORT_PATH.read_bytes()
source_report_sha256_before = hashlib.sha256(source_bytes).hexdigest()
report = json.loads(source_bytes.decode("utf-8"))
if report.get("schema") != repair.SCHEMA:
    raise RuntimeError(f"Unexpected report schema: {report.get('schema')}")
gate = report.get("candidate_selection_gate")
if gate != "HOLD":
    raise RuntimeError(f"Recovery requires exact HOLD report, got: {gate}")
if report.get("candidate_checkpoint_written") is not False:
    raise RuntimeError("HOLD report claims candidate checkpoint write")
if report.get("model_parameter_mutation_executed") is not False:
    raise RuntimeError("HOLD report claims model parameter mutation")
if report.get("gradient_based_model_training_executed") is not False:
    raise RuntimeError("HOLD report claims gradient training")
for key, expected in {
    "historical_validation_opened": False,
    "historical_retention_executed_by_this_module": False,
    "first30_opened": False,
    "v5_validation_opened": False,
    "final_holdout_locked": True,
    "digit4_frozen": True,
    "runtime_threshold_tuning": False,
}.items():
    if report.get(key) != expected:
        raise RuntimeError(f"Safety mismatch: {key}")

hold_reasons = []
for digit in ("2", "3"):
    specialist = report["per_specialist"][digit]
    fit = specialist["fit"]
    claim = fit.get("candidate_claim")
    copy_gate = specialist.get("float32_copy_gate")
    runtime_gate = specialist.get("runtime_float32_gate", "NOT_RUN_BECAUSE_SELECTION_GATE_HOLD")
    if claim != "CANDIDATE_WITNESS_VERIFIED":
        hold_reasons.append(f"{digit}-AI_CLAIM_{claim}")
    elif copy_gate is None:
        hold_reasons.append(f"{digit}-AI_FLOAT32_COPY_NOT_RUN")
    elif copy_gate.get("gate") != "PASS":
        hold_reasons.append(f"{digit}-AI_FLOAT32_COPY_{copy_gate.get('gate')}")
    print()
    print(f"========== {digit}-AI ==========")
    print("CLAIM =", claim)
    print("PRIMARY SOLVER =", {key: fit.get(key) for key in ("primary_status", "primary_success", "primary_iterations", "primary_optimality_claim")})
    print("SECONDARY SOLVER =", {key: fit.get(key) for key in ("secondary_status", "secondary_success", "secondary_iterations", "secondary_optimality_claim")})
    print("FLOAT32 COPY GATE =", copy_gate if copy_gate is not None else "NOT_RUN")
    print("RUNTIME FLOAT32 GATE =", runtime_gate)
    print("STATE INTEGRITY =", specialist.get("state_invariants", "NOT_RUN_BECAUSE_SELECTION_GATE_HOLD"))
    print("CANDIDATE =", specialist.get("candidate", "NOT_WRITTEN"))
    print("PATH DIAGNOSIS =", specialist.get("path_diagnosis"))

if not hold_reasons:
    hold_reasons.append("TOP_LEVEL_HOLD_WITHOUT_EXPOSED_SPECIALIST_REASON")
source_report_sha256_after = hashlib.sha256(REPORT_PATH.read_bytes()).hexdigest()
if source_report_sha256_after != source_report_sha256_before:
    raise RuntimeError("Source report changed during recovery")
envelope = {
    "schema": "st-omr-meter-v5-3a-hold-render-recovery-envelope-v1",
    "repository": REPOSITORY,
    "expected_execution_head": EXPECTED_EXECUTION_HEAD,
    "source_harness_head": SOURCE_HARNESS_HEAD,
    "source_implementation_ci_run_id": SOURCE_IMPLEMENTATION_CI_RUN_ID,
    "source_harness_ci_run_id": SOURCE_HARNESS_CI_RUN_ID,
    "recovered_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_report_path": str(REPORT_PATH),
    "source_report_sha256_before": source_report_sha256_before,
    "source_report_sha256_after": source_report_sha256_after,
    "source_report_preserved": True,
    "candidate_selection_gate": gate,
    "hold_reasons": hold_reasons,
    "report_recovery_only": True,
    "candidate_fit_reexecuted": False,
    "candidate_checkpoint_written": False,
    "gradient_training_executed": False,
    "historical_retention_executed": False,
    "first30_opened": False,
    "v5_validation_opened": False,
    "final_holdout_locked": True,
}
v51._atomic_write_json(RECOVERY_ENVELOPE_PATH, envelope)
recovery_sha256 = hashlib.sha256(RECOVERY_ENVELOPE_PATH.read_bytes()).hexdigest()
print()
print("============================================")
print("V5-3A HOLD REPORT RECOVERY RESULT")
print("============================================")
print("SOURCE REPORT PRESERVED = PASS")
print("CANDIDATE SELECTION GATE =", gate)
print("HOLD REASONS =", hold_reasons)
print("REPORT =", REPORT_PATH)
print("REPORT SHA256 =", source_report_sha256_after)
print("RECOVERY ENVELOPE =", RECOVERY_ENVELOPE_PATH)
print("RECOVERY ENVELOPE SHA256 =", recovery_sha256)
print("CANDIDATE FIT REEXECUTED = False")
print("CANDIDATE CHECKPOINT WRITTEN = False")
print("HISTORICAL RETENTION = NOT RUN")
print("FIRST-30 = CLOSED | V5 VAL = CLOSED | FINAL HOLDOUT = LOCKED")
